In [1]:
import rospy
from giskardpy.python_interface.python_interface import GiskardWrapper
import numpy as np
from geometry_msgs.msg import PoseStamped, Point, Quaternion, Vector3Stamped
from tf.transformations import quaternion_from_matrix, quaternion_about_axis
from giskardpy.goals.adaptive_goals import CloseGripper, PouringAdaptiveTilt

In [2]:
rospy.init_node('giskard_notebook')

In the next cell parameters of the pouring controller can be changed to explore its behaviour.
Further, one can have a look into the file ROS_WS/src/silkie_ros/src/silkie_ros/rules.dfl to have a look at or experiment with the rules composing the reasoning part of the controller.

In [3]:
# Set primary controller parameters
# The goal fill level for the second cup
rospy.set_param('pouring_controller/goal', 1)

# Set secondary controller parameters, like the velocity gains for translations and rotation
gain_translation = 0.02
gain_tilt_forward = 0.03
gain_tilt_backward = 1

# Set environment parameters to change the start conditions for the pouring controller
# Specifically, the position of the gripper holding the first cup from where the controller is started.
# This position is described in the coordinate frame of the second cup, which has the same orientation as the map coordinate frame
start_position = Point(-0.3, 0.2, 0.3)

The following 4 cells do
1. Restart the simulation. Success is indicated by the reset of the visualization.
2. Start or restart the Giskard node that controls the simulated robot.
3. Starts a script that analyzes the simulation data and publishes bounding boxes for all objects (not for the particles).
4. Starts a script that contains the reasoner that based on the rules and the current context decides the next best motion primitive to influence Giskard. Output can be seen in the topic /reasoner/concluded_behaviour.

Everytime one wants to repeat this experiment, the 4 commands should be executed again. Note that the goal parameter should be set before the reasoner is started.

In [4]:
%%bash --bg
roslaunch hsr_mujoco hsrb4s_velocity_two_cups_vrb.launch > /dev/null

In [5]:
%%bash --bg
roslaunch giskardpy giskardpy_hsr_mujoco.launch  > /dev/null

In [6]:
%%bash --bg
python3 ~/giskard_examples/ROS_WS/src/giskardpy/scripts/tools/mujoco_bb_detector.py -s > /dev/null 

In [7]:
%%bash --bg
cd ~/giskard_examples/ROS_WS/src/silkie_ros/src/silkie_ros && python3 pouring_reasoner.py  > /dev/null

In [8]:
gs = GiskardWrapper()

#### If the robot is not shown but the environment is, then click the reload button in rviz
The robot normally stands to the right of the table.

In [9]:
# this giskard code makes the robot move to the table, lets it grasp and lift the objects and then start the reasoning-based pouring controller.
# When pouring is done, the grasped cup is placed back onto the table.
goal_pose = PoseStamped()
goal_pose.header.frame_id = 'map'
goal_pose.pose.orientation = Quaternion(*quaternion_from_matrix([[1, 0, 0, 0],
                                                                 [0, 1, 0, 0],
                                                                 [0, 0, 1, 0],
                                                                 [0, 0, 0, 1]]))

goal_pose.pose.position.x = 1.0
goal_pose.pose.position.y = -0.2
goal_pose.pose.position.z = 0.0

gs.motion_goals.add_cartesian_pose(goal_pose, 'base_link', 'map')
gs.motion_goals.allow_all_collisions()
gs.add_default_end_motion_conditions()
gs.execute()

gs.motion_goals.add_motion_goal(motion_goal_class=CloseGripper.__name__,
                                name='openGripper',
                                as_open=True,
                                velocity_threshold=100,
                                effort_threshold=1,
                                effort=100)
gs.motion_goals.allow_all_collisions()
gs.add_default_end_motion_conditions()
gs.execute()

goal_pose = PoseStamped()
goal_pose.header.frame_id = 'map'
goal_pose.pose.orientation = Quaternion(*quaternion_from_matrix([[0, 0, 1, 0],
                                                                 [0, -1, 0, 0],
                                                                 [1, 0, 0, 0],
                                                                 [0, 0, 0, 1]]))
goal_pose.pose.position.x = 1.95
goal_pose.pose.position.y = -0.2
goal_pose.pose.position.z = 0.3

gs.motion_goals.add_cartesian_pose(goal_pose, 'hand_palm_link', 'map')
gs.motion_goals.allow_all_collisions()
gs.add_default_end_motion_conditions()
gs.execute()

gs.motion_goals.add_motion_goal(motion_goal_class=CloseGripper.__name__,
                                name='closeGripper', effort=-220)
gs.motion_goals.allow_all_collisions()
gs.add_default_end_motion_conditions()
gs.execute()

cup_pose = PoseStamped()
cup_pose.header.frame_id = 'free_cup'
cup_pose.pose.position = Point(0, 0, 0)
cup_pose.pose.orientation.w = 1

# add a new object at the pose of the pot and attach it to the right tip
gs.world.add_box('cup1', (0.07, 0.07, 0.28), pose=cup_pose, parent_link='hand_palm_link')
cup_pose.header.frame_id = 'free_cup2'
gs.world.add_box('cup2', (0.07, 0.07, 0.18), pose=cup_pose, parent_link='map')


goal_pose.header.frame_id = 'cup2'
goal_pose.pose.position = start_position
tilt_axis = Vector3Stamped()
tilt_axis.header.frame_id = 'hand_palm_link'
tilt_axis.vector.z = 1

tilt_angle = 0.6
# if the pouring starts from the right side of the second cup the first cup tilted to the left, otherwise it's tilted to the right
if goal_pose.pose.position.y < 0:
    tilt_angle *= -1

gs.motion_goals.add_motion_goal(motion_goal_class='PouringAdaptiveTilt',
                                name='pouring',
                                tip='hand_palm_link',
                                root='map',
                                tilt_angle=0.6,
                                pouring_pose=goal_pose,
                                tilt_axis=tilt_axis,
                                pre_tilt=False,
                                gain_translation=gain_translation,
                                gain_tilt_forward=gain_tilt_forward,
                                gain_tilt_backward=gain_tilt_backward)
gs.motion_goals.allow_all_collisions()
gs.motion_goals.avoid_collision(0.01, 'cup1', 'cup2')
gs.add_default_end_motion_conditions()
gs.execute()

goal_pose.header.frame_id = 'map'
goal_pose.pose.position.x = 1.93
goal_pose.pose.position.y = -0.2
goal_pose.pose.position.z = 0.3

gs.motion_goals.add_cartesian_pose(goal_pose, 'hand_palm_link', 'map')
gs.motion_goals.allow_all_collisions()
gs.add_default_end_motion_conditions()
gs.execute()

gs.motion_goals.add_motion_goal(motion_goal_class=CloseGripper.__name__,
                                name='openGripper',
                                as_open=True,
                                velocity_threshold=100,
                                effort_threshold=1,
                                effort=100)
gs.motion_goals.allow_all_collisions()
gs.add_default_end_motion_conditions()
gs.execute()

goal_pose.pose.position.x = 1.4
goal_pose.pose.position.y = -0.2
goal_pose.pose.position.z = 0.4

gs.motion_goals.add_cartesian_pose(goal_pose, 'hand_palm_link', 'map')
gs.motion_goals.allow_all_collisions()
gs.add_default_end_motion_conditions()
gs.execute()
print('Done')

Done
